In [59]:
# Step 1: Import Modules and Utility Functions
import importlib
import BitcoinNewsQA_utils
importlib.reload(BitcoinNewsQA_utils)

from haystack.document_stores import InMemoryDocumentStore
from haystack.nodes import BM25Retriever, FARMReader
from haystack.pipelines import ExtractiveQAPipeline
from BitcoinNewsQA_utils import (
    fetch_crypto_news,
    create_documents,
    split_documents,
    analyze_sentiment
)

import random


Device set to use mps:0


In [61]:
# Step 2: Fetch and Prepare News Articles
news = fetch_crypto_news()
print(f"Fetched {len(news)} articles")

documents = create_documents(news)
chunks = split_documents(documents)
print(f"Prepared {len(chunks)} chunks")


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/priyamchhonkar/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Fetched 20 articles


Preprocessing: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 94.22docs/s]

Prepared 20 chunks


In [63]:
# Step 3: Create Document Store and Write Documents
document_store = InMemoryDocumentStore(use_bm25=True)
document_store.write_documents(chunks)


Updating BM25 representation...: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 22215.59 docs/s]


In [65]:
# Step 4: Setup Retriever and Reader
retriever = BM25Retriever(document_store=document_store)
reader = FARMReader(model_name_or_path="deepset/roberta-base-squad2", use_gpu=False)

# Create the QA pipeline
pipe = ExtractiveQAPipeline(reader=reader, retriever=retriever)


In [71]:
# Step 5: Define List of Bitcoin-Related Questions
questions = [
    "Why did Bitcoin rise this week?",
    "Which company recently added Bitcoin to its treasury?",
    "What’s Basel Medical Group doing with Bitcoin?",
    "Why is BlackRock in the news related to Bitcoin?",
    "Is Bitcoin seen as a safe haven this week?",
    "What is the SEC hacker case about?",
    "Why are pension funds buying Bitcoin ETFs?",
    "How is Asia influencing Bitcoin markets now?",
    "Who is warning about Bitcoin volatility now?",
    "What’s the media saying about Bitcoin’s next move?"
]


In [78]:
# Step 6: Select and Ask a Random Question
query = random.choice(questions)
print("Question:", query)

result = pipe.run(query=query, params={"Retriever": {"top_k": 5}, "Reader": {"top_k": 1}})
answer = result['answers'][0]

# Step 7: Display Answer, Context, Source, and Sentiment
print("Answer:", answer.answer)
print("Context:", answer.context)
print("Source:", answer.meta.get("source", "N/A"))

# Analyze sentiment of the answer context
label, score = analyze_sentiment(answer.context)
print("Sentiment:", label, f"(Confidence Score: {score:.2f})")


Question: Which company recently added Bitcoin to its treasury?


Inferencing Samples: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.23 Batches/s]


Answer: Basel Medical
Context: $1B Bitcoin Deal: Basel Medical’s Unconventional Treasury Strategy - https://cryptopanic.com/news/21869276/1B-Bitcoin-Deal-Basel-Medicals-Unconvention
Source: 99bitcoins.com
Sentiment: NEGATIVE (Confidence Score: 1.00)
